In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import *

spark = SparkSession.builder.getOrCreate()

data1 = [("USA", 2018, 20544.34), ("USA", 2019, 21427.7), ("China", 2018, 13894.04)]
data2 = [("China", 2019, 14402.72), ("India", 2018, 2713.61), ("India", 2019, 2868.93)]
cols = ["Country", "Year", "GDP"]

df1 = spark.createDataFrame(data1, cols)
df2 = spark.createDataFrame(data2, cols)

df1.show()
df2.show()

+-------+----+--------+
|Country|Year|     GDP|
+-------+----+--------+
|    USA|2018|20544.34|
|    USA|2019| 21427.7|
|  China|2018|13894.04|
+-------+----+--------+

+-------+----+--------+
|Country|Year|     GDP|
+-------+----+--------+
|  China|2019|14402.72|
|  India|2018| 2713.61|
|  India|2019| 2868.93|
+-------+----+--------+



In [5]:
df = df1.union(df2)

# Define window partitioned by Country and ordered by Year
window = Window.partitionBy("Country").orderBy("Year")

# Compute GDP of previous year and add as a new column
df = df.withColumn("prev_year_GDP", lag("GDP").over(window))

# Calculate the GDP growth rate
df = df.withColumn(
    "GDP_growth_rate",
    when(isnull(df.prev_year_GDP), None).otherwise(
        ((df.GDP - df.prev_year_GDP) / df.prev_year_GDP) * 100
    ),
)

# Select only necessary columns and round off GDP_growth_rate to 2 decimal places
df = df.select(
    "Country",
    "Year",
    round(df.GDP_growth_rate, 2).alias("GDP_growth_rate"),
)
df.show()

+-------+----+---------------+
|Country|Year|GDP_growth_rate|
+-------+----+---------------+
|  China|2018|           NULL|
|  China|2019|           3.66|
|  India|2018|           NULL|
|  India|2019|           5.72|
|    USA|2018|           NULL|
|    USA|2019|            4.3|
+-------+----+---------------+

